In [1]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
from peft import PeftModel

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla P100-PCIE-16GB


## 1. Load Trained Model

In [ ]:
ADAPTER_DIR = "/kaggle/input/models/jiayuanshe/akka2eng-lora-02/transformers/default/1"
BASE_MODEL_DIR = "/kaggle/input/models/jiayuanshe/byt5-large-base/transformers/default/1"

assert os.path.exists(ADAPTER_DIR), f"Adapter dir not found: {ADAPTER_DIR}"
assert os.path.exists(BASE_MODEL_DIR), f"Base model dir not found: {BASE_MODEL_DIR}"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, local_files_only=True)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_DIR,
    local_files_only=True
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    local_files_only=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print(f"Adapter loaded from: {ADAPTER_DIR}")
print(f"Base model loaded from: {BASE_MODEL_DIR}")
print(f"Running on device: {device}")

Loading weights:   0%|          | 0/498 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Adapter loaded from: /kaggle/input/models/jiayuanshe/akka2eng-lora-02/transformers/default/1
Base model loaded from: /kaggle/input/models/jiayuanshe/byt5-large-base/transformers/default/1
Running on device: cuda


## 2. Load Test Data

In [3]:
# Load test data
TEST_DATA_DIR = "/kaggle/input/competitions/deep-past-initiative-machine-translation/"
test_csv_path = os.path.join(TEST_DATA_DIR, "test.csv")

assert os.path.exists(test_csv_path), f"Test data not found: {test_csv_path}"

test_df = pd.read_csv(test_csv_path)

print(f"Test data shape: {test_df.shape}")
print(f"Test columns: {list(test_df.columns)}")
print(f"Sample test data:")
print(test_df.head())

Test data shape: (4, 5)
Test columns: ['id', 'text_id', 'line_start', 'line_end', 'transliteration']
Sample test data:
   id   text_id  line_start  line_end  \
0   0  332fda50           1         7   
1   1  332fda50           7        14   
2   2  332fda50          14        24   
3   3  332fda50          25        30   

                                     transliteration  
0  um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-t...  
1  i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-n...  
2  ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na...  
3  me-+e-er mup-pì-ni a-na kà-ar kà-ar-ma ú wa-ba...  


## 3. Prepare Test Inputs

In [4]:
# Prepare test inputs with task prefix
TASK_PREFIX = "Translate Akkadian transliteration to English: "
MAX_SRC_LEN = 384
MAX_TGT_LEN = 192

# Get source column (transliteration)
SRC_COL = "transliteration"
assert SRC_COL in test_df.columns, f"Column '{SRC_COL}' not found in test data"

# Prepare test texts
test_texts = (TASK_PREFIX + test_df[SRC_COL].astype(str)).tolist()

print(f"Total test samples: {len(test_texts)}")
print(f"\nSample input:")
print(test_texts[0][:100] + "...")

Total test samples: 4

Sample input:
Translate Akkadian transliteration to English: um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-tim aí-i...


## 4. Generate Predictions

In [5]:
def generate_translations(texts, batch_size=16, max_new_tokens=MAX_TGT_LEN):
    """
    Generate translations for a list of texts using the trained model.
    
    Args:
        texts: list of input texts
        batch_size: batch size for inference
        max_new_tokens: max length of generated tokens
    
    Returns:
        list of generated translations
    """
    all_preds = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating predictions"):
        batch_texts = texts[i:i + batch_size]
        
        # Tokenize batch
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SRC_LEN,
        ).to(device)
        
        # Generate
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams=4,
                early_stopping=True,
            )
        
        # Decode
        predictions = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )
        
        all_preds.extend(predictions)
    
    return all_preds

# Generate predictions
print("Generating predictions for test set...")
predictions = generate_translations(test_texts, batch_size=16)

print(f"\nGenerated {len(predictions)} predictions")
print(f"\nSample predictions:")
for i in range(min(3, len(predictions))):
    print(f"[{i}] {predictions[i][:100]}...")

Generating predictions for test set...


Generating predictions:   0%|          | 0/1 [00:00<?, ?it/s]


Generated 4 predictions

Sample predictions:
[0] Thus says the karum of Kanesh: 'I have sent the message to the kanesh and the kanesh and the wabarra...
[1] From the message of the city from this month, anyone who has brought the tin to me at this time will...
[2] As you have written to me, whether it is for the city or for the city of the city, or for the city o...


## 5. Create Submission File

In [6]:
# Create submission dataframe
submission_df = pd.DataFrame({
    "id": test_df["id"].values,
    "translation": predictions
})

print(f"Submission shape: {submission_df.shape}")
print(f"\nSubmission preview:")
print(submission_df.head(10))

Submission shape: (4, 2)

Submission preview:
   id                                        translation
0   0  Thus says the karum of Kanesh: 'I have sent th...
1   1  From the message of the city from this month, ...
2   2  As you have written to me, whether it is for t...
3   3  I have sent my tablets to the city of the city...


## 6. Save Submission

In [7]:
# Save submission
SUBMISSION_PATH = "./submission.csv"
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"Submission saved to: {SUBMISSION_PATH}")
print(f"File size: {os.path.getsize(SUBMISSION_PATH) / 1024:.2f} KB")

# Verify submission
print(f"\nVerification:")
print(f"Total rows: {len(submission_df)}")
print(f"Columns: {list(submission_df.columns)}")
print(f"No missing translations: {submission_df['translation'].notna().all()}")

Submission saved to: ./submission.csv
File size: 0.53 KB

Verification:
Total rows: 4
Columns: ['id', 'translation']
No missing translations: True


## 7. Compare with Sample Submission

In [8]:
# Load sample submission for format verification
SAMPLE_SUBMISSION_PATH = os.path.join(TEST_DATA_DIR, "sample_submission.csv")
sample_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print(f"Sample submission shape: {sample_df.shape}")
print(f"Sample submission columns: {list(sample_df.columns)}")
print(f"\nSample submission preview:")
print(sample_df.head())

print(f"\n✓ Our submission format matches sample format")
print(f"✓ Both have {len(submission_df)} rows")

Sample submission shape: (4, 2)
Sample submission columns: ['id', 'translation']

Sample submission preview:
   id                                        translation
0   0  Thus  Kanesh, say to the -payers, our messenge...
1   1  In the letter of the City (it is written): Fro...
2   2  As soon as you have heard our letter, who(ever...
3   3  Send a copy of (this) letter of ours to every ...

✓ Our submission format matches sample format
✓ Both have 4 rows


## 8. Show Some Examples

In [9]:
# Show some example translations
print("Sample Translations:")
print("=" * 100)

for idx in range(min(5, len(test_df))):
    print(f"\n[ID: {test_df.iloc[idx]['id']}]")
    print(f"Source (Akkadian): {test_df.iloc[idx]['transliteration']}")
    print(f"Generated Translation: {submission_df.iloc[idx]['translation']}")
    print("-" * 100)

Sample Translations:

[ID: 0]
Source (Akkadian): um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-tim aí-ip-ri-ni kà-ar kà-ar-ma ú wa-bar-ra-tim qí-bi„-ma mup-pu-um aa a-lim(ki) i-li-kam
Generated Translation: Thus says the karum of Kanesh: 'I have sent the message to the kanesh and the kanesh and the wabarratum to you.'
----------------------------------------------------------------------------------------------------

[ID: 1]
Source (Akkadian): i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-nim ma-ma-an KÙ.AN i-aa-ú-mu-ni i-na né-mì-lim da-aùr ú-lá e-WA ia-ra-tí-au kà-ru-um kà-ni-ia i-lá-qé
Generated Translation: From the message of the city from this month, anyone who has brought the tin to me at this time will take the karum of my kanesh.
----------------------------------------------------------------------------------------------------

[ID: 2]
Source (Akkadian): ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na aí-mì-im a-na É.GAL-lim i-dí-in lu té-ra-at É.GAL-lim ú-kà-lim lu na-aí-ma a